# NoSQL·Redis 환경 확인

> **`db-pg`는 PostgreSQL 실습 때부터 띄워 둔 컨테이너를 오늘도 그대로 이어서 씁니다.** `db-redis`는 오늘 처음 필요한 컨테이너라 아직 없을 수 있는데, 그 경우엔 이 노트북이 자동으로 만들어 줍니다. `PostgreSQL_1_Docker`처럼 절차를 하나하나 밟는 노트북이 아니라, 두 컨테이너가 잘 준비돼 있는지 **확인**하는 짧은 노트북입니다.

## 오늘 확인할 것
- `db-pg`(PostgreSQL, course_db) 컨테이너가 켜져 있는지 · `db-redis` 컨테이너가 켜져 있는지(없으면 새로 생성)
- PostgreSQL·Redis에 실제로 연결되는지
- MongoDB는 오늘도 mongomock(pymongo와 거의 동일한 API로 동작하는 실제 파이썬 라이브러리)으로 대체합니다 — 별도 컨테이너가 필요 없어 설치 확인만 합니다

## 실행 안내
- Docker 컨테이너 기동 확인은 Python `subprocess`로 작성해 **Windows·Linux·macOS 어디서나 동일하게 동작**합니다(셸 문법 차이가 없습니다).
- PostgreSQL·Redis 연결 확인은 각각 `psycopg`·`redis-py` 파이썬 라이브러리로 직접 연결합니다 — `psql`·`redis-cli` 같은 별도 커맨드라인 도구를 이 컴퓨터에 따로 설치할 필요가 없습니다(설치돼 있지 않으면 `FileNotFoundError`가 납니다). PostgreSQL 비밀번호는 코드에 쓰지 않고 같은 폴더 `.env` 파일(`PGPASSWORD=postgres`)에서 `load_dotenv()`로 읽습니다.
- 위에서 아래로 순서대로 실행하세요. 확인이 모두 끝나면 `NoSQL_Redis_2_핵심개념_데모.ipynb`로 넘어가세요.

In [1]:
%pip install -q mongomock redis "psycopg[binary]==3.3.4" python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Docker 컨테이너 기동 확인

In [1]:
# ✅ db-pg는 시작만, db-redis는 없으면 오늘 처음으로 새로 만듭니다.
# 💡 db-pg는 9/16부터 계속 이어온 컨테이너라 start만 하면 되지만, db-redis는 오늘(9/23) 처음 필요한
#    컨테이너라 아직 없을 수 있습니다 — docker inspect로 존재 여부를 먼저 확인하고, 없을 때만 새로 만듭니다
#    (재실행 안전: 이미 있으면 이 단계를 건너뜁니다).
import subprocess  # 파이썬에서 터미널 명령(docker, psql 등)을 대신 실행해주는 표준 라이브러리

# docker inspect db-redis: 그 이름의 컨테이너 정보를 조회합니다 — 없으면 returncode가 0이 아닙니다.
inspect = subprocess.run(["docker", "inspect", "db-redis"], capture_output=True, text=True)
if inspect.returncode != 0:
    print("db-redis 컨테이너가 아직 없어 새로 만듭니다 (redis:8, 포트 6379:6379)...")
    # -d(백그라운드) · --name db-redis(이름) · -p 6379:6379(포트) · redis:8(이미지)
    # 💡 볼륨(-v) 옵션이 없습니다 — 오늘 실습은 인메모리 학습용이라 컨테이너를 지우면 데이터도 함께 사라집니다.
    create = subprocess.run(
        ["docker", "run", "-d", "--name", "db-redis", "-p", "6379:6379", "redis:8"],
        capture_output=True, text=True,
    )
    print(create.stdout.strip() if create.returncode == 0 else create.stderr.strip())
else:
    print("db-redis 컨테이너가 이미 있습니다 — 재사용합니다.")

# subprocess.run(명령어_리스트, ...)는 터미널에 그 명령어를 친 것과 같습니다.
# 리스트의 각 항목이 공백으로 구분된 한 단어씩입니다 — ["docker", "start", "db-pg", "db-redis"]는
# 터미널의 `docker start db-pg db-redis`와 완전히 같은 명령입니다.
# capture_output=True : 명령어가 화면에 찍을 출력을 파이썬 변수로 "붙잡아서" 가져옵니다(안 하면 그냥 사라짐).
# text=True            : 그 출력을 바이트(b'...')가 아니라 우리가 읽기 편한 일반 문자열(str)로 받습니다.
start = subprocess.run(["docker", "start", "db-pg", "db-redis"], capture_output=True, text=True)

# subprocess.run()의 결과(start)는 세 가지를 담고 있습니다.
#  - start.returncode : 명령어가 성공하면 0, 실패하면 0이 아닌 숫자(리눅스 명령어의 공통 관례)
#  - start.stdout     : 명령어가 정상 출력한 내용(표준 출력)
#  - start.stderr     : 명령어가 오류 시 출력한 내용(표준 오류)
# .strip()은 줄바꿈(\n)이나 공백이 앞뒤에 붙어 있으면 잘라내는 문자열 메서드입니다 — 출력을 깔끔하게 보려고 씁니다.
print("기동 확인:", start.stdout.strip() if start.returncode == 0 else start.stderr.strip())

# docker ps는 "현재 떠 있는 컨테이너 목록"을 보여주는 명령어입니다.
# --format "{{.Names}}\t{{.Status}}"는 출력 형식을 "이름<탭>상태"만 나오게 줄인 것입니다(Docker 고유 템플릿 문법).
ps = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}\t{{.Status}}"],
    capture_output=True, text=True,
)
print(ps.stdout)
# 기대: db-pg, db-redis 두 줄이 "Up ..." 상태로 보여야 합니다.


db-redis 컨테이너가 아직 없어 새로 만듭니다 (redis:8, 포트 6379:6379)...
d1ffbe6b0a3c9da57e64c30c6c5094ea3365abfc6c5ee57c50bc0477184ec9e7
기동 확인: db-pg
db-redis
db-redis	Up Less than a second
db-neo4j	Up 2 minutes
db-pg	Up 5 minutes



## 2. PostgreSQL·Redis 연결 확인

In [2]:
# ✅ PostgreSQL(course_db) 연결 확인 — `PostgreSQL_1_Docker` 이후 지금까지와 같은 psycopg + .env 방식입니다.
# 💡 포인트: 비밀번호를 코드에 쓰지 않습니다 — 같은 폴더의 .env 파일에 두고 load_dotenv()로 읽습니다.
#    .env 내용은 PGPASSWORD=postgres 처럼 이름=값 한 줄씩 — Git에 올리지 않는 파일입니다!
import os               # 환경변수(비밀번호 등)를 읽는 표준 라이브러리
import psycopg          # Python용 PostgreSQL 드라이버 (v3 — import psycopg2가 아닙니다)
from dotenv import load_dotenv

# ✅ 포인트: psycopg.connect(host, port, dbname, user, password)로 실제로 연결합니다.
#    psql 같은 별도 커맨드라인 도구가 이 컴퓨터에 설치돼 있지 않아도, 이 파이썬 코드만으로 연결됩니다.
# 💡 이 노트북은 "확인만" 하는 노트북이라, .env 파일이 없거나 연결이 안 되더라도 에러로 멈추지 않고 아래
#    Redis·mongomock 확인까지 이어서 실행되도록 아래 전체를 try/except로 감쌌습니다.
try:
    load_dotenv()                        # 같은 폴더의 .env → 환경변수로 등록
    pw = os.environ["PGPASSWORD"]        # 비밀번호는 .env 파일에 (하드코딩 금지) — 파일이 없으면 여기서 KeyError
    conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)
    cur = conn.cursor()
    cur.execute("SELECT 1;")   # 실제 데이터가 필요 없어 연결 자체가 되는지만 확인할 때 관례적으로 쓰는 SQL
    cur.fetchone()
    print("PostgreSQL 연결: OK →", conn)
    cur.close()
    conn.close()               # 확인용 연결이라 바로 닫습니다 — 다음 시간(NoSQL_Redis_2)에서 새로 연결합니다.
except Exception as e:
    print("PostgreSQL 연결 실패:", e)

PostgreSQL 연결: OK → <psycopg.Connection [INTRANS] (host=localhost user=postgres database=course_db) at 0x1ad52ec2270>


In [3]:
# ✅ Redis 연결 확인 — redis-cli가 이 컴퓨터(호스트)에 따로 설치돼 있지 않아도 되도록, redis-py로 직접 연결합니다.
# 💡 redis-cli는 Docker Desktop과 별개로 이 컴퓨터 자체에 설치해야 하는 도구라, 없으면 FileNotFoundError가 납니다.
#    redis-py는 pip로만 설치되는 파이썬 라이브러리라 이 문제가 없고, NoSQL_Redis_4_기본_자료구조.ipynb부터
#    실제로 쓰는 것과 같은 클라이언트이기도 합니다.
import redis

# 💡 이 노트북은 "확인만" 하는 노트북이라, 연결이 안 되더라도 에러로 멈추지 않고 아래 mongomock 확인까지
#    이어서 실행되도록 try/except로 감쌌습니다.
try:
    # redis.Redis(...)는 Redis 서버에 접속하는 "연결 객체"를 만듭니다 — 아직 실제로 통신하지는 않습니다.
    r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
    pong = r.ping()   # 여기서 실제로 서버에 요청을 보냅니다 — 살아있으면 True (redis-cli의 PONG과 같은 뜻)
    print("Redis 연결:", "PONG" if pong else "응답 없음")
except Exception as e:
    print("Redis 연결 실패:", e)

Redis 연결: PONG


## 3. mongomock 설치 확인

In [4]:
# ✅ mongomock — 실제 서버가 아니라 인메모리 시뮬레이션이지만, pymongo와 거의 동일한 API로 동작하는 실제 파이썬 라이브러리입니다.
import mongomock  # pip로 설치하는 외부 패키지 — "진짜 MongoDB" 대신 파이썬 메모리 안에서 흉내 내주는 라이브러리

print("mongomock 버전:", mongomock.__version__)  # __version__은 대부분의 파이썬 라이브러리가 갖는 버전 표시 변수

# MongoClient()는 "MongoDB 서버에 접속하는 창구"를 만드는 과정입니다.
# 진짜 pymongo였다면 실제 네트워크로 서버에 접속하지만, mongomock은 접속 없이 메모리 안에서 그 역할을 흉내 냅니다.
mongo_client = mongomock.MongoClient()
# mongo_client["course_db"]처럼 대괄호에 이름을 넣으면 그 이름의 데이터베이스를 "선택"합니다
# (SQL의 USE course_db;와 비슷한 개념 — 실제로 데이터를 만들지는 않고, 이후 작업의 대상만 지정).
# .name은 방금 선택한 데이터베이스의 이름을 다시 문자열로 돌려줍니다 — 잘 선택됐는지 확인용입니다.
print("MongoClient 생성 확인:", mongo_client["course_db"].name)


mongomock 버전: 4.3.0
MongoClient 생성 확인: course_db


In [5]:
!docker ps

CONTAINER ID   IMAGE                     COMMAND                   CREATED          STATUS          PORTS                                                                                      NAMES
d1ffbe6b0a3c   redis:8                   "docker-entrypoint.s…"   24 minutes ago   Up 24 minutes   0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp                                                db-redis
470f1c8e37cf   neo4j:2026.06-community   "tini -g -- /startup…"   4 days ago       Up 27 minutes   0.0.0.0:7474->7474/tcp, [::]:7474->7474/tcp, 0.0.0.0:7687->7687/tcp, [::]:7687->7687/tcp   db-neo4j
b9045c5dac14   pgvector/pgvector:pg17    "docker-entrypoint.s…"   6 days ago       Up 30 minutes   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp                                                db-pg


## 확인 완료

- [ ] `docker ps`에 `db-pg`·`db-redis`가 모두 `Up` 상태로 보였다
- [ ] PostgreSQL 연결이 `OK`였다
- [ ] Redis 연결이 `PONG`이었다
- [ ] mongomock 버전이 출력됐다

모두 확인됐다면 `NoSQL_Redis_2_핵심개념_데모.ipynb`로 넘어가세요. 컨테이너가 안 떠 있다면(예: `db-pg`가 목록에 없음) 9/16 실습 자료의 Docker 안내를 참고해 다시 띄운 뒤 이 노트북을 재실행하세요.
